In [1]:
import numpy as np
import pandas as pd
import torch
from torch import nn

import selfies as sf
device = "cuda"

In [2]:
# import os
# os.chdir('~/code/selfies/examples/vae_example/')
#from chemistry_vae import VAEDecoder, VAEEncoder

In [3]:
def get_smiles_encodings_for_dataset(file_path):
    
    df = pd.read_csv(file_path)

    smiles_list = np.asanyarray(df.smiles)

    smiles_alphabet = list(set(''.join(smiles_list)))
    smiles_alphabet.append(' ')  # for padding
    largest_smiles_len = len(max(smiles_list, key=len))

    return smiles_list, smiles_alphabet, largest_smiles_len

def get_selfies_encodings_for_dataset(smiles_list, largest_smiles_len):

    print("Largest smiles len", largest_smiles_len)
    print('--> Translating SMILES to SELFIES...')
    selfies_list = list(map(sf.encoder, smiles_list))
    f = open("datasets/0SelectedSMILES_QM9.sf.txt", "w")
    f.write("\n".join(selfies_list))
    all_selfies_symbols = sf.get_alphabet_from_selfies(selfies_list)
    all_selfies_symbols.add('[nop]')
    selfies_alphabet = list(all_selfies_symbols)

    largest_selfies_len = max(sf.len_selfies(s) for s in selfies_list)
    print("Largest selfies len", largest_selfies_len)

    print('Finished translating SMILES to SELFIES.')
    return selfies_list, selfies_alphabet, largest_selfies_len

def selfies_to_hot(selfie, largest_selfie_len, alphabet):
    """Go from a single selfies string to a one-hot encoding.
    """

    symbol_to_int = dict((c, i) for i, c in enumerate(alphabet))

    # pad with [nop]
    selfie += '[nop]' * (largest_selfie_len - sf.len_selfies(selfie))

    # integer encode
    symbol_list = sf.split_selfies(selfie)
    integer_encoded = [symbol_to_int[symbol] for symbol in symbol_list]

    # one hot-encode the integer encoded selfie
    onehot_encoded = list()
    for index in integer_encoded:
        letter = [0] * len(alphabet)
        letter[index] = 1
        onehot_encoded.append(letter)

    return integer_encoded, np.array(onehot_encoded)



def multiple_selfies_to_hot(selfies_list, largest_molecule_len, alphabet):
    """Convert a list of selfies strings to a one-hot encoding
    """

    hot_list = []
    for s in selfies_list:
        _, onehot_encoded = selfies_to_hot(s, largest_molecule_len, alphabet)
        hot_list.append(onehot_encoded)
    return np.array(hot_list)

# def load_models(epoch):
#     print("loading models")
#     out_dir = './saved_models/{}'.format(epoch)
#     encoder = torch.load('{}/E'.format(out_dir), map_location=torch.device(device))
#     encoder.eval()
#     decoder = torch.load('{}/D'.format(out_dir), map_location=torch.device(device))
#     decoder.eval()
#     return encoder, decoder
  

In [4]:
# get all the inputs
num = -1
smiles_list, smiles_alphabet, largest_smiles_len = get_smiles_encodings_for_dataset("datasets/0SelectedSMILES_QM9.txt")
selfies_list, selfies_alphabet, largest_selfies_len = get_selfies_encodings_for_dataset(smiles_list[:num], largest_smiles_len)


Largest smiles len 22
--> Translating SMILES to SELFIES...
Largest selfies len 21
Finished translating SMILES to SELFIES.


In [5]:
print(selfies_list[0])

[C]


In [6]:

def selfie_to_integers(selfie, symbol_to_int, max_len):
    """Convert to a sequence of integers. Add a "start token" and "end token".
    """
    symbol_list = list(sf.split_selfies(selfie))
    return [len(symbol_to_int)] + [symbol_to_int[symbol] for symbol in symbol_list] + [len(symbol_to_int) + 1]

def selfies_to_integers(selfies, alphabet, max_len):
    symbol_to_int = dict((c, i) for i, c in enumerate(alphabet))
    return [selfie_to_integers(selfie, symbol_to_int, max_len) for selfie in selfies]

selfies_ints = selfies_to_integers(selfies_list, selfies_alphabet, largest_selfies_len)

In [22]:
symbol_to_int = dict((c, i) for i, c in enumerate(selfies_alphabet))
selfie_to_integers(selfies_list[0], symbol_to_int, largest_selfies_len)

[18, 16, 19]

In [7]:
selfies_list[0]

'[C]'

In [8]:
import torch
from torch.utils.data import Dataset
from torch.nn.utils.rnn import pad_sequence
from functools import partial

class SelfiesDataset(Dataset):
    def __init__(self, data_list):
        self.data = [torch.tensor(item).to(device) for item in data_list]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

def collate_fn(batch, *, pad_idx):
    return pad_sequence(batch, batch_first=True, padding_value=pad_idx)

pad_idx = selfies_alphabet.index('[nop]')
dataset = SelfiesDataset(selfies_ints, )

data_loader = torch.utils.data.DataLoader(
    dataset, batch_size=32, shuffle=True,
    collate_fn=partial(collate_fn, pad_idx=selfies_alphabet.index('[nop]'))
)


In [9]:
import pytorch_lightning as pl
from torchmetrics.classification import MulticlassAccuracy


class VAEEncoder(nn.Module):
    def __init__(
        self,
        n_vocab: int,
        pad_idx: int,
        q_dim: int = 512,
        q_layers: int = 1,
        latent_dim: int = 128,
        emb_dim: int = 256,
    ):
        super().__init__()
        self.pad_idx = pad_idx
        self.n_vocab = n_vocab
        self.q_dim = q_dim
        self.q_layers = q_layers

        # Embeddings layer to avoid the need to one hot encode manually
        self.embed = nn.Embedding(n_vocab, emb_dim, self.pad_idx)
        self.encoder_rnn = nn.GRU(
            emb_dim,
            self.q_dim,
            num_layers=self.q_layers,
            batch_first=True,
            bidirectional=False,
        )
        self.q_mu = nn.Linear(self.q_dim, latent_dim)
        self.q_logvar = nn.Linear(self.q_dim, latent_dim)

    def forward(self, x: torch.Tensor) -> dict[str, torch.Tensor]:
        x_emb = self.embed(x)
        h_complete, hidden = self.encoder_rnn(x_emb)
        # batch first, and use only last layer
        hidden = hidden.permute(1,0,2)[:,:1,:]
        mu, log_var = self.q_mu(hidden), self.q_logvar(hidden)
        return mu, log_var, x_emb, hidden, h_complete


class VAEDecoder(nn.Module):
    def __init__(
        self,
        n_vocab: int,
        d_dim: int = 512,
        d_layers: int = 1,
        latent_dim: int = 128,
    ):
        super().__init__()
        self.n_vocab = n_vocab
        self.d_dim = d_dim
        self.d_layers = d_layers

        self.decoder_rnn = nn.GRU(
            latent_dim,
            self.d_dim,
            num_layers=self.d_layers,
            batch_first=True,
        )
        self.decoder_fc = nn.Linear(self.d_dim, n_vocab)

    def forward(self, x: torch.Tensor, hidden: torch.Tensor=None) -> torch.Tensor:
        output, hidden = self.decoder_rnn(x)
        return self.decoder_fc(output), hidden



class VAE(pl.LightningModule):
    def __init__(
        self,
        n_vocab: int,
        pad_idx: int,
        q_dim: int = 512,
        q_layers: int = 1,
        latent_dim: int = 128,
        emb_dim: int = 256,
        d_dim:int=512,
        d_layers:int=1,
        lr:float=1e-4
    ):
        super().__init__()

        self.lr = lr
        self.encoder = VAEEncoder(n_vocab=n_vocab, pad_idx=pad_idx,emb_dim=emb_dim,q_dim=q_dim,q_layers=q_layers,latent_dim=latent_dim)
        self.decoder = VAEDecoder(
            n_vocab=n_vocab,latent_dim=latent_dim,
            d_dim=d_dim,d_layers=d_layers
        )
        self.accuracy = MulticlassAccuracy(
            n_vocab,
        )
        self.cross_entropy = nn.CrossEntropyLoss(
            reduction="mean",
        )

    def reparametize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        noise = torch.randn_like(std).to(self.device)
        z = mu + noise * std
        return z

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.lr)

    def training_step(self, batch, batch_idx):
        x = batch

        mu, log_var, x_emb, hidden, h_complete = self.encoder(x)

        # reparameterization trick
        std = torch.exp(log_var / 2.)
        q = torch.distributions.Normal(mu, std)
        z = q.rsample()

        # This is the same as doing a for loop and passing the hidden state
        x_dec, _ = self.decoder(z.repeat(1,x.size(1),1))

        # reconstruction loss

        recon_loss = self.cross_entropy(x_dec.permute(0,2,1), x).mean()
        kl_loss = torch.mean(
            -0.5 * torch.sum(1 + log_var - mu**2 - log_var.exp(), dim=1), dim=0
        ).mean()

        elbo = (kl_loss - recon_loss)

        # could add a weight for kl
        loss = (recon_loss + kl_loss)
        accuracy = self.accuracy(x_dec.argmax(-1), x)

        self.log_dict({
            'loss': loss,
            'elbo': elbo,
            'kl': kl_loss,
            'recon_loss': recon_loss,
            'accuracy': accuracy,
        })

        return loss


C:\Users\davidek\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
vae = VAE(
    len(selfies_alphabet) + 2,
    selfies_alphabet.index('[nop]'),
    q_dim = 512,
    q_layers = 3,
    latent_dim = 512,
    emb_dim = 512,
    d_dim = 512,
    d_layers = 3, 
    lr = 2e-4
).to(device)


In [11]:
from lightning.pytorch.loggers import WandbLogger
from lightning.pytorch import Trainer
import wandb



data_loader = torch.utils.data.DataLoader(
    dataset, batch_size=128, shuffle=True,
    collate_fn=partial(collate_fn, pad_idx=selfies_alphabet.index('[nop]'))
)

# wandb.init()
# wandb_logger = WandbLogger(
#     project="selfies_vae",
# )
trainer = pl.Trainer(devices=1, accelerator='cuda', max_epochs=20)#, logger=wandb_logger)
trainer.fit(vae, data_loader)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
C:\Users\davidek\AppData\Local\Programs\Python\Python312\Lib\site-packages\pytorch_lightning\trainer\connectors\logger_connector\logger_connector.py:75: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `pytorch_lightning` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
You are using a CUDA device ('NVIDIA GeForce RTX 4090') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.h

Epoch 19: 100%|██████████████████████████████████████████████████████████| 1032/1032 [00:13<00:00, 74.45it/s, v_num=11]

`Trainer.fit` stopped: `max_epochs=20` reached.


Epoch 19: 100%|██████████████████████████████████████████████████████████| 1032/1032 [00:15<00:00, 67.93it/s, v_num=11]


In [12]:
# Example decoding
idx = 0
selfies_ints[idx]

[18, 16, 19]

In [14]:
# make it a tensor and fix input dimensions
input_tensor = torch.nn.functional.pad(
    torch.Tensor(selfies_ints[idx]).unsqueeze(0).long(),
    pad=(0,largest_selfies_len-len(selfies_ints[idx])),
    value=selfies_alphabet.index('[nop]')
)
latent, *_ = vae.encoder(input_tensor)
latent

tensor([[[-3.7548e+00, -9.2283e-02,  2.6732e-01,  4.1118e-02,  6.5886e-02,
          -2.4296e+00, -2.0675e-01,  1.9592e-01,  1.3327e-01,  6.4035e-01,
           1.9506e-01,  1.3808e-01, -4.3970e-01, -2.7581e-01,  2.1224e-01,
          -5.7391e-01,  1.0098e-01, -1.0503e-02, -3.1920e-02, -9.8434e-02,
           2.3369e-01, -5.1391e-01, -4.1480e-01, -1.5326e-01, -2.3625e-01,
           1.3203e-01,  7.1873e-02,  3.6725e-01,  8.1143e-02, -1.1646e-01,
           3.5790e-01, -5.9493e-01,  5.3211e-03,  8.5416e-01,  3.4909e-01,
           3.2764e-01,  6.6749e-02, -2.3432e-01,  1.0280e-03, -1.4573e+00,
          -2.9733e-01, -8.5015e-02,  2.2150e-01,  5.3514e-02, -1.4915e-01,
          -1.1830e-01,  2.0871e-01, -3.8737e-01, -5.2346e-02, -1.0770e-01,
           3.9596e-01,  7.8229e-02, -4.7398e-01,  7.5689e-01, -2.3730e-01,
           6.7461e-02,  1.8370e-01, -5.1607e-01,  1.7825e-01,  4.2115e-01,
           8.6792e-02, -9.4443e-02,  4.5987e-01, -6.6936e-03,  8.7539e-02,
           3.9122e-02, -1

In [15]:
# Since we don't a priori know the length, just use something longer than the training data
decoded, _ = vae.decoder(latent.repeat(1,largest_selfies_len,1))
# back to indices
decoded.argmax(-1)


tensor([[18, 16, 19, 19,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,
          5,  5,  5]])

In [17]:
vae.accuracy(input_tensor, decoded.argmax(-1))

tensor(0.8750)

In [23]:
result = []
for x in decoded.argmax(-1).tolist()[0]:
    try:
        result.append(selfies_alphabet[x])
    except IndexError:
        pass
print(''.join(result))

[C][nop][nop][nop][nop][nop][nop][nop][nop][nop][nop][nop][nop][nop][nop][nop][nop][nop]


In [29]:
sf.decoder('[N][N][=C][C][=N][Ring1][Branch1]')
sf.decoder('[C][C][C][Branch1][C][O][C][#N]')
sf.decoder('[C][C][C][=Branch1][C][=O][C][C]')

'CCC(=O)CC'

In [40]:
mol1 = '[N][N][=C][C][=N][Ring1][Branch1]'
mol2 = '[C][C][C][=Branch1][C][=O][C][C]'
idx1 = selfies_list.index(mol1)
idx2 = selfies_list.index(mol2)

In [42]:
# make it a tensor and fix input dimensions
input_tensor = torch.nn.functional.pad(
    torch.Tensor(selfies_ints[idx1]).unsqueeze(0).long(),
    pad=(0,largest_selfies_len-len(selfies_ints[idx1])),
    value=selfies_alphabet.index('[nop]')
)
latent1, *_ = vae.encoder(input_tensor)
latent1

tensor([[[-2.1353e+00,  2.8029e-02,  1.0865e-01,  1.7014e-01, -9.1635e-02,
          -5.3272e-01, -1.0500e-01,  4.0687e-02, -4.6341e-02,  1.0590e-01,
           1.4538e-01,  1.3415e-01, -4.1428e-02, -1.2254e-01, -1.0106e-01,
          -1.0369e-01, -1.1732e-02, -1.9467e-03, -1.7458e-02,  5.3790e-02,
          -1.2391e-01, -3.5902e-02,  4.5929e-02,  2.1257e-01,  3.7382e-02,
          -3.4446e-02,  6.5043e-02,  2.9057e-02,  9.2966e-02,  4.4025e-02,
           2.9057e-02, -3.5379e-01, -3.0547e-02,  1.2044e+00, -1.7082e-02,
           2.4256e-02, -9.5077e-02,  3.2435e-01,  7.6770e-02, -2.0057e+00,
          -1.7024e-01, -1.4569e-01, -1.5957e-01,  1.6676e-01, -1.9178e-01,
          -8.5614e-02, -1.3232e-01,  1.1233e-02, -5.1544e-02, -6.9872e-02,
           7.3965e-02,  3.4906e-02, -1.3135e-01,  4.9567e-01,  8.7895e-02,
          -2.7938e-02, -9.4548e-02, -1.8844e-02, -3.1876e-02, -4.9998e-02,
          -6.6859e-03, -1.8947e-01, -1.3352e-03, -1.7844e-01,  6.8209e-02,
          -8.4280e-02,  5

In [43]:
# make it a tensor and fix input dimensions
input_tensor = torch.nn.functional.pad(
    torch.Tensor(selfies_ints[idx2]).unsqueeze(0).long(),
    pad=(0,largest_selfies_len-len(selfies_ints[idx2])),
    value=selfies_alphabet.index('[nop]')
)
latent2, *_ = vae.encoder(input_tensor)
latent2

tensor([[[-1.9470e+00,  1.6882e-02,  2.8164e-02,  1.3605e-01,  1.0981e-01,
          -1.0860e+00, -2.7237e-02, -7.9160e-02,  4.6325e-02, -3.9979e-03,
           7.1719e-02,  6.4266e-02, -3.8626e-02, -3.8412e-02, -3.5512e-03,
          -2.2470e-01, -1.1674e-01,  9.6260e-03,  9.6190e-02,  1.0918e-01,
           1.2512e-01,  5.9901e-04, -8.4051e-02,  1.8613e-02, -1.0292e-01,
           1.6422e-02,  4.9873e-02, -5.1692e-02, -3.6204e-02,  1.1144e-02,
          -4.2708e-02, -3.1794e-01,  1.1237e-01, -9.5514e-01,  1.6218e-02,
          -8.4001e-02, -2.8309e-03, -1.4219e+00, -1.2161e-01, -1.6086e+00,
          -1.7648e-01,  1.2651e-02, -6.7152e-02, -4.2048e-02, -3.8259e-02,
          -1.5174e-02, -1.4009e-01, -1.7770e-01, -1.6576e-01, -4.1458e-02,
           2.4462e-02, -4.4814e-02, -1.8445e-01, -4.1077e-02, -1.9999e-02,
          -1.0754e-01, -5.4393e-02, -3.0625e-03,  4.2776e-02, -1.9036e-02,
          -6.0223e-02, -8.2718e-02,  8.5988e-02, -1.2364e-01,  1.0050e-01,
           7.5372e-02, -6

In [54]:
points = torch.linspace(latent1, latent2, 50)

RuntimeError: linspace only supports 0-dimensional start and end tensors, but got start with 3 dimension(s) and end with 3 dimension(s).

In [56]:
for point in points:
    decoded, _ = vae.decoder(torch.from_numpy(point).repeat(1,largest_selfies_len,1))
    # back to indices
    
    decoded.argmax(-1)
    result = []
    for x in decoded.argmax(-1).tolist()[0]:
        try:
            result.append(selfies_alphabet[x])
        except IndexError:
            pass
    print(''.join(result))


[N][C][=N][C][=N][Ring1][Branch1][nop][nop][nop][nop][nop][nop][nop][nop][nop][nop][nop][nop]
[N][C][=N][C][=N][Ring1][Branch1][nop][nop][nop][nop][nop][nop][nop][nop][nop][nop][nop][nop]
[N][C][=N][C][=N][Ring1][Branch1][nop][nop][nop][nop][nop][nop][nop][nop][nop][nop][nop][nop]
[N][C][=N][C][=N][Ring1][Branch1][nop][nop][nop][nop][nop][nop][nop][nop][nop][nop][nop][nop]
[N][C][=N][C][=N][Ring1][Branch1][nop][nop][nop][nop][nop][nop][nop][nop][nop][nop][nop][nop]
[N][C][=N][C][=N][Ring1][Branch1][nop][nop][nop][nop][nop][nop][nop][nop][nop][nop][nop][nop]
[N][C][=N][C][=N][Ring1][Branch1][nop][nop][nop][nop][nop][nop][nop][nop][nop][nop][nop][nop]
[N][C][=C][C][=N][Ring1][Branch1][nop][nop][nop][nop][nop][nop][nop][nop][nop][nop][nop][nop]
[N][C][=C][C][=N][Ring1][Branch1][nop][nop][nop][nop][nop][nop][nop][nop][nop][nop][nop][nop]
[N][C][=C][C][=N][Ring1][Branch1][nop][nop][nop][nop][nop][nop][nop][nop][nop][nop][nop][nop]
[N][C][=C][C][=N][Ring1][Branch1][nop][nop][nop][nop][nop][n